# Optuna hyperparameter search (PPO + MlpPolicy, Lunar Lander)

Search space is loaded from `optuna_ppo_search_space_lunarlander_v3.json` at run time. This notebook uses **50 trials** and **1,500,000** environment steps per trial. Network width/depth is **fixed** (same as `lunar_rl_common.policy_kwargs`); only PPO and training hyperparameters are searched.

Results are written to `best_hyperparams.json` (or whatever path you set in the settings cell).

**Requirement:** run with the working directory set to the repo root `RL-LunarLander`, or set `OUTPUT` to an absolute path.


In [ ]:
from __future__ import annotations

import copy
import json
import os
import tempfile

import optuna
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

from lunar_rl_common import (
    EntropyCoefScheduleCallback,
    make_ent_coef_schedule_late_linear,
    make_eval_vec_env_with_stats,
    make_ppo_lr_schedule_late_linear,
    make_train_vec_env,
    policy_kwargs as default_policy_kwargs,
    resolve_train_device,
    suggested_parallel_envs,
)

# --- Search space spec (mirrors optuna_ppo_search_space_lunarlander_v3.json) ---
_REPO_ROOT = os.getcwd()
_SEARCH_SPACE_PATH = os.path.join(_REPO_ROOT, "optuna_ppo_search_space_lunarlander_v3.json")
if not os.path.isfile(_SEARCH_SPACE_PATH):
    _SEARCH_SPACE_PATH = "optuna_ppo_search_space_lunarlander_v3.json"

with open(_SEARCH_SPACE_PATH, encoding="utf-8") as _f:
    _SPEC = json.load(_f)

_fixed = _SPEC["fixed_params"]
_SEARCH = _SPEC["search_space"]

# --- Run settings ---
N_TRIALS = 50
TIMESTEPS_PER_TRIAL = 1_500_000
N_EVAL_EPISODES = 10
N_ENVS = _fixed.get("n_envs")
if N_ENVS is None:
    N_ENVS = suggested_parallel_envs()
SEED = int(_fixed.get("seed", 42))
OUTPUT = "best_hyperparams.json"
ENV_ID = _fixed.get("env_id", "LunarLander-v3")
STUDY_NAME = _SPEC.get("study_name", "ppo-lunarlander-v3-focused-search")
DEVICE_PREF = str(_fixed.get("train_device", "cpu")).lower()

_ACTIVATION = {"Tanh": torch.nn.Tanh, "ReLU": torch.nn.ReLU}


def _valid_batch_sizes(n_steps: int, n_envs: int, base_choices: list) -> list[int]:
    total = int(n_steps) * int(n_envs)
    out: list[int] = []
    for b in base_choices:
        if b <= total and total % b == 0:
            out.append(int(b))
    return out


def suggest_ppo_hyperparams(trial: optuna.Trial, n_envs: int) -> dict:
    """Map Optuna trial to hyperparams using search_space from the JSON spec."""
    ss = _SEARCH

    n_steps = trial.suggest_categorical("n_steps", ss["n_steps"]["choices"])
    base_bs = ss["batch_size"]["base_choices"]
    valid_bs = _valid_batch_sizes(n_steps, n_envs, base_bs)
    if not valid_bs:
        raise optuna.TrialPruned()

    lr_spec = ss["learning_rate_start"]
    lr_start = trial.suggest_float(
        "learning_rate_start", lr_spec["low"], lr_spec["high"], log=lr_spec.get("log", True)
    )
    lr_end = trial.suggest_categorical("learning_rate_end", ss["learning_rate_end"]["choices"])
    flat_p = trial.suggest_categorical(
        "schedule_flat_until_progress", ss["schedule_flat_until_progress"]["choices"]
    )

    batch_size = trial.suggest_categorical("batch_size", valid_bs)
    n_epochs = trial.suggest_categorical("n_epochs", ss["n_epochs"]["choices"])
    gamma = trial.suggest_categorical("gamma", ss["gamma"]["choices"])
    gae_lambda = trial.suggest_categorical("gae_lambda", ss["gae_lambda"]["choices"])
    clip_range = trial.suggest_categorical("clip_range", ss["clip_range"]["choices"])

    ent_spec = ss["ent_coef_start"]
    ent_start = trial.suggest_float(
        "ent_coef_start", ent_spec["low"], ent_spec["high"], log=ent_spec.get("log", True)
    )
    ent_end = trial.suggest_categorical("ent_coef_end", ss["ent_coef_end"]["choices"])
    target_kl = trial.suggest_categorical("target_kl", ss["target_kl"]["choices"])
    vf_coef = trial.suggest_categorical("vf_coef", ss["vf_coef"]["choices"])
    max_grad_norm = trial.suggest_categorical("max_grad_norm", ss["max_grad_norm"]["choices"])

    act_name = trial.suggest_categorical("activation_fn", ss["activation_fn"]["choices"])
    ortho_init = trial.suggest_categorical("ortho_init", ss["ortho_init"]["choices"])

    return {
        "n_steps": n_steps,
        "batch_size": batch_size,
        "n_epochs": n_epochs,
        "learning_rate_start": lr_start,
        "learning_rate_end": lr_end,
        "schedule_flat_until_progress": flat_p,
        "gamma": gamma,
        "gae_lambda": gae_lambda,
        "clip_range": clip_range,
        "ent_coef_start": ent_start,
        "ent_coef_end": ent_end,
        "target_kl": target_kl,
        "vf_coef": vf_coef,
        "max_grad_norm": max_grad_norm,
        "activation_fn": _ACTIVATION[act_name],
        "activation_fn_name": act_name,
        "ortho_init": bool(ortho_init),
    }


def hp_for_json(hp: dict) -> dict:
    """JSON-serializable copy (no torch classes). Includes fixed net_arch for logging."""
    out = {k: v for k, v in hp.items() if k != "activation_fn"}
    out["activation_fn"] = hp["activation_fn_name"]
    out["net_arch"] = default_policy_kwargs["net_arch"]
    return out


device = resolve_train_device(DEVICE_PREF)
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(
    f"device={device}, n_envs={N_ENVS}, timesteps/trial={TIMESTEPS_PER_TRIAL:,}, "
    f"n_trials={N_TRIALS}, spec={_SEARCH_SPACE_PATH!r}, output={OUTPUT!r}"
)


In [ ]:
def objective(trial: optuna.Trial) -> float:
    hp = suggest_ppo_hyperparams(trial, N_ENVS)

    trial_env = make_train_vec_env(N_ENVS, SEED, hp["gamma"], env_id=ENV_ID)

    lr_sched = make_ppo_lr_schedule_late_linear(
        lr_start=hp["learning_rate_start"],
        lr_end=hp["learning_rate_end"],
        flat_until_progress=hp["schedule_flat_until_progress"],
    )
    ent_sched = make_ent_coef_schedule_late_linear(
        ent_start=hp["ent_coef_start"],
        ent_end=hp["ent_coef_end"],
        flat_until_progress=hp["schedule_flat_until_progress"],
    )
    ent_cb = EntropyCoefScheduleCallback(ent_sched)

    policy_kwargs_trial = copy.deepcopy(default_policy_kwargs)
    policy_kwargs_trial["activation_fn"] = hp["activation_fn"]
    policy_kwargs_trial["ortho_init"] = hp["ortho_init"]

    model = PPO(
        "MlpPolicy",
        trial_env,
        seed=SEED,
        device=device,
        verbose=0,
        policy_kwargs=policy_kwargs_trial,
        learning_rate=lr_sched,
        clip_range=hp["clip_range"],
        clip_range_vf=None,
        normalize_advantage=_fixed.get("normalize_advantage", True),
        vf_coef=hp["vf_coef"],
        max_grad_norm=hp["max_grad_norm"],
        use_sde=_fixed.get("use_sde", False),
        target_kl=hp["target_kl"],
        n_steps=hp["n_steps"],
        batch_size=hp["batch_size"],
        n_epochs=hp["n_epochs"],
        gamma=hp["gamma"],
        gae_lambda=hp["gae_lambda"],
        ent_coef=hp["ent_coef_start"],
    )

    model.learn(
        total_timesteps=TIMESTEPS_PER_TRIAL,
        callback=[ent_cb],
    )

    trial_vec_path = os.path.join(
        tempfile.gettempdir(), f"optuna_vecnormalize_trial_{trial.number}.pkl"
    )
    trial_env.save(trial_vec_path)
    trial_env.close()

    eval_venv = make_eval_vec_env_with_stats(trial_vec_path, SEED, env_id=ENV_ID)
    mean_reward, std_reward = evaluate_policy(
        model,
        eval_venv,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
    )
    eval_venv.close()
    try:
        os.remove(trial_vec_path)
    except OSError:
        pass
    del model

    score = mean_reward - std_reward
    trial.set_user_attr("mean_reward", mean_reward)
    trial.set_user_attr("std_reward", std_reward)
    trial.set_user_attr("resolved_params", hp_for_json(hp))
    return float(score)


print(
    f"Starting Optuna study: {N_TRIALS} trials, "
    f"{TIMESTEPS_PER_TRIAL:,} timesteps each, {N_ENVS} envs"
)
study = optuna.create_study(direction="maximize", study_name=STUDY_NAME)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_data = {
    "params": study.best_trial.params,
    "params_resolved": study.best_trial.user_attrs.get("resolved_params"),
    "score": study.best_trial.value,
    "mean_reward": study.best_trial.user_attrs["mean_reward"],
    "std_reward": study.best_trial.user_attrs["std_reward"],
    "trial_number": study.best_trial.number,
    "search_space_file": os.path.abspath(_SEARCH_SPACE_PATH),
    "study_name": STUDY_NAME,
    "run": {
        "seed": SEED,
        "n_envs": N_ENVS,
        "timesteps_per_trial": TIMESTEPS_PER_TRIAL,
        "n_eval_episodes": N_EVAL_EPISODES,
        "env_id": ENV_ID,
    },
}
with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(best_data, f, indent=2)

print(f"\nBest trial #{study.best_trial.number}:")
print(f"  Score (mean - std): {study.best_trial.value:.2f}")
print(f"  Mean reward:        {study.best_trial.user_attrs['mean_reward']:.2f}")
print(f"  Std reward:         {study.best_trial.user_attrs['std_reward']:.2f}")
print("  Optuna params:")
for k, v in study.best_trial.params.items():
    print(f"    {k}: {v}")
rp = study.best_trial.user_attrs.get("resolved_params")
if rp:
    print("  Resolved (schedules + policy overrides; net_arch fixed):")
    for k, v in rp.items():
        print(f"    {k}: {v}")
print(f"\nSaved to {OUTPUT!r}")
